In [30]:
import random
import pandas as pd
import math
from datetime import datetime, timedelta

def random_location():
    lat = random.uniform(6.4, 6.6)
    lon = random.uniform(3.3, 3.5)
    return lat, lon

def interpolate_point(start, end, fraction):
    lat = start[0] + fraction * (end[0] - start[0])
    lon = start[1] + fraction * (end[1] - start[1])
    return lat, lon

def generate_gps_trace(pickup_time, trip_duration, cancel_delay, true_label):
    start_location = random_location()
    interval_seconds = 30
    gps_trace = []

    if true_label == 'genuine_cancel':
        parked_steps = (cancel_delay * 60) // interval_seconds
        for step in range(parked_steps + 1):
            lat = start_location[0] + random.uniform(-0.0002, 0.0002)
            lon = start_location[1] + random.uniform(-0.0002, 0.0002)
            timestamp = pickup_time + timedelta(seconds=step * interval_seconds)
            gps_trace.append({'time': timestamp, 'lat': lat, 'lon': lon})

        away_minutes = random.randint(3, 10)
        away_steps = (away_minutes * 60) // interval_seconds
        speed_kmh = random.uniform(15, 35)
        angle = random.uniform(0, 2 * math.pi)
        step_deg = (speed_kmh / 111) * (interval_seconds / 3600)
        for k in range(1, away_steps + 1):
            lat = start_location[0] + k * step_deg * math.sin(angle)
            lon = start_location[1] + k * step_deg * math.cos(angle)
            timestamp = pickup_time + timedelta(seconds=(parked_steps + k) * interval_seconds)
            gps_trace.append({'time': timestamp, 'lat': lat, 'lon': lon})
        return gps_trace

    speed_kmh = random.uniform(20, 40)
    distance_km = speed_kmh * (trip_duration / 60)
    distance_deg = distance_km / 111
    angle = random.uniform(0, 2 * math.pi)
    end_location = (
        start_location[0] + distance_deg * math.sin(angle),
        start_location[1] + distance_deg * math.cos(angle)
    )

    total_seconds = trip_duration * 60
    num_steps = total_seconds // interval_seconds
    for step in range(num_steps + 1):
        fraction = step / num_steps
        point = interpolate_point(start_location, end_location, fraction)
        timestamp = pickup_time + timedelta(seconds=step * interval_seconds)
        gps_trace.append({'time': timestamp, 'lat': point[0], 'lon': point[1]})

    if true_label == 'fraud_cancel_silent':
        cancel_time = pickup_time + timedelta(minutes=cancel_delay)
        end_time = pickup_time + timedelta(minutes=trip_duration)
        back_on = end_time - timedelta(minutes=2)
        gps_trace = [p for p in gps_trace
                     if p['time'] <= cancel_time or p['time'] >= back_on]
    return gps_trace

def make_drivers(n_drivers=50, fraud_share=0.1):
    drivers = {}
    for driver_id in range(1, n_drivers + 1):
        if random.random() < fraud_share:
            drivers[driver_id] = {'fraudster': True, 'style': random.choice(['naive', 'silent'])}
        else:
            drivers[driver_id] = {'fraudster': False, 'style': None}
    return drivers

def generate_trip(trip_id, driver_id, profile):
    if profile['fraudster']:
        labels = ['completed', 'genuine_cancel', 'fraud_cancel_' + profile['style']]
        weights = [65, 10, 25]
    else:
        labels = ['completed', 'genuine_cancel']
        weights = [90, 10]
    true_label = random.choices(labels, weights=weights, k=1)[0]

    date = datetime(2026, 1, 1, 0, 0, 0)
    r = random.randint(0, 1439)
    pickup_time = date + timedelta(minutes=r)

    trip_duration = random.randint(5, 40)

    if true_label == 'completed':
        status = 'completed'
        cancel_delay = None
        cancel_time = None
    else:
        status = 'cancelled'
        cancel_delay = random.randint(1, 15)
        cancel_time = pickup_time + timedelta(minutes=cancel_delay)
    if true_label in ('fraud_cancel_naive', 'fraud_cancel_silent'):
        trip_duration = random.randint(cancel_delay + 5, 40)

    gps_trace = generate_gps_trace(pickup_time, trip_duration, cancel_delay, true_label)

    return {
        'trip_id': trip_id,
        'driver_id': driver_id,
        'status': status,
        '_true_label': true_label,
        'pickup_time': pickup_time,
        'cancel_time': cancel_time,
        'gps_trace': gps_trace
    }

silent = generate_gps_trace(datetime(2026, 1, 1, 8, 0, 0), 20, 4, 'fraud_cancel_silent')
print(len(silent), silent[8]['time'], silent[9]['time'])
genuine = generate_gps_trace(datetime(2026, 1, 1, 8, 0, 0), 20, 4, 'genuine_cancel')
print(genuine[0]['time'], genuine[-1]['time'])

14 2026-01-01 08:04:00 2026-01-01 08:18:00
2026-01-01 08:00:00 2026-01-01 08:07:00


In [31]:
def generate_dataset(n_trips=2000, n_drivers=50):
    drivers = make_drivers(n_drivers)
    trips = []
    for trip_id in range(1, n_trips + 1):
        driver_id = random.randint(1, n_drivers)
        trips.append(generate_trip(trip_id, driver_id, drivers[driver_id]))
    return trips

In [32]:
trips = generate_dataset()
df = pd.DataFrame(trips)
print(df.shape)
print(df['status'].value_counts())
print(df['_true_label'].value_counts())

(2000, 7)
status
completed    1750
cancelled     250
Name: count, dtype: int64
_true_label
completed              1750
genuine_cancel          213
fraud_cancel_silent      37
Name: count, dtype: int64


In [33]:
df['trace_len'] = df['gps_trace'].apply(len)
print(df.groupby('_true_label')['trace_len'].mean())

_true_label
completed              46.330286
fraud_cancel_silent    20.594595
genuine_cancel         29.666667
Name: trace_len, dtype: float64


In [34]:
df.to_pickle('trips.pkl')

In [35]:
silent = generate_gps_trace(datetime(2026, 1, 1, 8, 0, 0), 20, 4, 'fraud_cancel_silent')
print(len(silent))
print(silent[8]['time'], silent[9]['time'])

14
2026-01-01 08:04:00 2026-01-01 08:18:00
